In [1]:
!pip install scikit-learn nltk pandas

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     --------------------------------- ------ 51.2/61.0 kB 2.7 MB/s eta 0:00:01
     --------------------------------- ------ 51.2/61.0 kB 2.7 MB/s eta 0:00:01
     --------------------------------- ------ 51.2/61.0 kB 2.7 MB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 294.8 kB/s eta 0:00:00
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 2.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ---------------------------------------- 57.7/57.7 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
    --------------------------------------- 0.1/8.1 MB 6.8 MB/s eta 0:00:02
   - -------------------------------------- 0.2/8.1 MB 2.8 MB/s eta 0:00:03
   - -------------------------------------- 0.4/8.1 MB 3.0 MB/s eta 0:00:03
   -- -


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully ✅")

Libraries loaded successfully ✅


In [9]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

print("Loading dataset...")
df = pd.read_csv("spam.csv", encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})
df['cleaned'] = df['message'].apply(preprocess_text)

print(f"Dataset loaded: {df.shape[0]} messages")
print(f"Spam: {(df['label'] == 'spam').sum()}")
print(f"Ham:  {(df['label'] == 'ham').sum()}")

Loading dataset...
Dataset loaded: 5572 messages
Spam: 747
Ham:  4825


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label_encoded'],
    test_size=0.2, random_state=42, stratify=df['label_encoded']
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

svm_model = LinearSVC(C=1.0, max_iter=10000, class_weight='balanced', random_state=42)
svm_model.fit(X_train_tfidf, y_train)

y_pred = svm_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"ACCURACY: {accuracy * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

ACCURACY: 98.21%
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       966
        Spam       0.96      0.91      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [11]:
with open("svm_model.pkl", 'wb') as f:
    pickle.dump(svm_model, f)
print("svm_model.pkl saved! ✅")

with open("svm_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)
print("svm_vectorizer.pkl saved! ✅")

svm_model.pkl saved! ✅
svm_vectorizer.pkl saved! ✅
